# AML/TM Databricks One-Stop Learning Notebook

Run this notebook top to bottom in Azure Databricks, Fabric Spark notebooks, or a Jupyter environment with PySpark. It consolidates the Databricks-related learning path into one runnable flow: platform mental model, widgets, Spark SQL, PySpark, DQ checks, reconciliation, Delta-style persistence, Databricks SQL outputs, Lakeflow/Jobs thinking, and ML feature readiness.

## What This Notebook Covers

This notebook uses a tiny public-safe AML/TM modernization case study:

```text
legacy extracts -> bronze raw data -> silver standardized data -> gold rule input
               -> DQ exceptions + reconciliation -> alerts + supporting transactions
               -> Databricks SQL / BI views -> job evidence -> feature-ready analytics
```

The data is synthetic. No private customer, bank, screenshot, credential, or proprietary rule logic is used.

## Step 0 - Databricks-Style Parameters and Helpers

Databricks notebooks often use widgets for job parameters. This cell uses widgets when `dbutils` exists and falls back to plain Python values outside Databricks.

In [ ]:
from __future__ import annotations

import shutil
from pathlib import Path

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder.appName("aml-databricks-one-stop-learning").getOrCreate()

def get_dbutils_or_none():
    try:
        return dbutils
    except NameError:
        return None

dbutils_ref = get_dbutils_or_none()

if dbutils_ref is not None:
    dbutils_ref.widgets.text("processing_month", "2022-06")
    dbutils_ref.widgets.text("rule_threshold_cad", "100")
    processing_month = dbutils_ref.widgets.get("processing_month")
    rule_threshold_cad = float(dbutils_ref.widgets.get("rule_threshold_cad"))
else:
    processing_month = "2022-06"
    rule_threshold_cad = 100.0

month_start = f"{processing_month}-01"
month_end = spark.sql(f"SELECT add_months(DATE '{month_start}', 1) AS month_end").first().month_end.isoformat()
batch_id = f"aml_tm_demo_{processing_month.replace('-', '')}"

def assert_set(name, actual_rows, expected_rows):
    actual = set(actual_rows)
    expected = set(expected_rows)
    assert actual == expected, f"{name}: expected {expected}, got {actual}"

def show_df(df, n=20):
    try:
        display(df)
    except NameError:
        df.show(n, truncate=False)

print(f"processing_month={processing_month}")
print(f"month_start={month_start}, month_end={month_end}")
print(f"rule_threshold_cad={rule_threshold_cad}")
print(f"batch_id={batch_id}")

## Step 1 - Create Synthetic Source Extracts

In a real Databricks project these rows might arrive from files, ADF/Fabric Data Factory, Lakeflow Connect, or upstream exports. Here they are created in-memory so the notebook is self-contained.

In [ ]:
transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("account_id", T.StringType(), True),
    T.StructField("transaction_date", T.StringType(), False),
    T.StructField("amount_cad", T.StringType(), False),
    T.StructField("transaction_type", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("country_code", T.StringType(), True),
])

source_transactions = spark.createDataFrame([
    ("t1", "a1", "2022-06-01", "60.00", "WIRE", "POSTED", "IR"),
    ("t2", "a1", "2022-06-03", "50.00", "WIRE", "POSTED", "IR"),
    ("t3", "a1", "2022-06-05", "10.00", "CARD", "POSTED", "CA"),
    ("t4", "a2", "2022-06-02", "200.00", "WIRE", "POSTED", "CA"),
    ("t5", "a3", "2022-06-02", "20.00", "WIRE", "REVERSED", "IR"),
    ("t6", "a9", "2022-06-02", "80.00", "WIRE", "POSTED", "IR"),
    ("t7", "a2", "2022-07-01", "300.00", "WIRE", "POSTED", "IR"),
    ("t8", "a4", "2022-06-10", "100.00", "CASH", "POSTED", None),
], schema=transaction_schema)

accounts = spark.createDataFrame([
    ("a1", "c1", "ACTIVE", "CHECKING"),
    ("a2", "c2", "ACTIVE", "CHECKING"),
    ("a3", "c3", "ACTIVE", "SAVINGS"),
    ("a4", "c4", "CLOSED", "CHECKING"),
], ["account_id", "customer_id", "account_status", "product_type"])

country_risk = spark.createDataFrame([
    ("IR", "HIGH"),
    ("CA", "LOW"),
    ("US", "LOW"),
], ["country_code", "risk_level"])

assert source_transactions.count() == 8
assert accounts.count() == 4
assert country_risk.count() == 3
show_df(source_transactions.orderBy("transaction_id"))

## Step 2 - Bronze Layer: Preserve Raw Shape and Add Run Metadata

Bronze should preserve what arrived while adding enough metadata to replay and audit the run.

In [ ]:
bronze_transactions = (
    source_transactions
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("source_system", F.lit("legacy_tm_extract"))
    .withColumn("ingested_at_utc", F.current_timestamp())
)

bronze_manifest = spark.createDataFrame([
    (batch_id, "transactions", bronze_transactions.count(), processing_month),
    (batch_id, "accounts", accounts.count(), processing_month),
    (batch_id, "country_risk", country_risk.count(), processing_month),
], ["batch_id", "dataset_name", "row_count", "processing_month"])

assert bronze_manifest.count() == 3
show_df(bronze_manifest.orderBy("dataset_name"))

## Step 3 - Silver Layer: Normalize Types and Standardize Keys

Silver data should be typed, standardized, and ready for DQ checks. This is where string dates become dates, numeric strings become decimals, and keys are normalized.

In [ ]:
silver_transactions = (
    bronze_transactions
    .withColumn("transaction_date", F.to_date("transaction_date", "yyyy-MM-dd"))
    .withColumn("amount_cad", F.col("amount_cad").cast("decimal(18,2)"))
    .withColumn("account_id", F.upper(F.trim("account_id")))
    .withColumn("country_code", F.upper(F.trim("country_code")))
    .withColumn("transaction_type", F.upper(F.trim("transaction_type")))
    .withColumn("status", F.upper(F.trim("status")))
)

assert silver_transactions.count() == 8
assert dict(silver_transactions.dtypes)["transaction_date"] == "date"
assert dict(silver_transactions.dtypes)["amount_cad"] == "decimal(18,2)"
show_df(silver_transactions.orderBy("transaction_id"))

## Step 4 - DQ Checks: Find Exceptions Before Rule Execution

A Databricks pipeline should not silently lose bad rows. Separate DQ exceptions from valid rule input and reconcile the counts.

In [ ]:
required_field_failures = silver_transactions.filter(
    F.col("transaction_id").isNull()
    | F.col("account_id").isNull()
    | F.col("transaction_date").isNull()
    | F.col("amount_cad").isNull()
)

orphan_accounts = silver_transactions.join(accounts, on="account_id", how="left_anti")

invalid_country = silver_transactions.filter(F.col("country_code").isNotNull()).join(
    country_risk,
    on="country_code",
    how="left_anti",
)

duplicate_transaction_ids = (
    silver_transactions.groupBy("transaction_id")
    .agg(F.count("*").alias("duplicate_count"))
    .filter(F.col("duplicate_count") > 1)
)

closed_account_transactions = (
    silver_transactions.join(accounts, on="account_id", how="inner")
    .filter(F.col("account_status") == "CLOSED")
)

dq_summary = spark.createDataFrame([
    ("required_field_failures", required_field_failures.count()),
    ("orphan_accounts", orphan_accounts.count()),
    ("invalid_country", invalid_country.count()),
    ("duplicate_transaction_ids", duplicate_transaction_ids.count()),
    ("closed_account_transactions", closed_account_transactions.count()),
], ["dq_check", "failed_row_count"])

assert required_field_failures.count() == 0
assert_set("orphan_accounts", [r.transaction_id for r in orphan_accounts.select("transaction_id").collect()], ["t6"])
assert invalid_country.count() == 0
assert duplicate_transaction_ids.count() == 0
assert_set("closed_account_transactions", [r.transaction_id for r in closed_account_transactions.select("transaction_id").collect()], ["t8"])
show_df(dq_summary.orderBy("dq_check"))

## Step 5 - Gold Rule Input: Create the Governed Rule-Ready Grain

The rule grain here is one eligible transaction joined to a valid customer/account and country-risk reference row.

In [ ]:
june_posted_wires = (
    silver_transactions
    .filter(F.col("status") == "POSTED")
    .filter(F.col("transaction_type") == "WIRE")
    .filter((F.col("transaction_date") >= F.lit(month_start)) & (F.col("transaction_date") < F.lit(month_end)))
)

valid_customer_tx = june_posted_wires.join(accounts, on="account_id", how="inner")

gold_rule_input = (
    valid_customer_tx.join(country_risk, on="country_code", how="left")
    .withColumn("processing_month", F.lit(processing_month))
    .withColumn("rule_id", F.lit("TM_HIGH_RISK_WIRE_001"))
    .withColumn("rule_version", F.lit("1.0.0"))
)

assert june_posted_wires.count() == 4
assert valid_customer_tx.count() == 3
assert gold_rule_input.count() == 3
show_df(gold_rule_input.orderBy("transaction_id"))

## Step 6 - Spark SQL and PySpark Are Two Front Doors

Register temp views so the same rule-ready data can be queried with Spark SQL. This mirrors how Databricks teams often mix PySpark transformations and SQL validation.

In [ ]:
gold_rule_input.createOrReplaceTempView("gold_rule_input")

sql_customer_totals = spark.sql(f"""
SELECT
  customer_id,
  SUM(amount_cad) AS observed_amount_cad,
  COUNT(*) AS supporting_transaction_count
FROM gold_rule_input
WHERE risk_level = 'HIGH'
GROUP BY customer_id
HAVING SUM(amount_cad) > {rule_threshold_cad}
""")

pyspark_customer_totals = (
    gold_rule_input.filter(F.col("risk_level") == "HIGH")
    .groupBy("customer_id")
    .agg(
        F.sum("amount_cad").alias("observed_amount_cad"),
        F.count("*").alias("supporting_transaction_count"),
    )
    .filter(F.col("observed_amount_cad") > F.lit(rule_threshold_cad))
)

assert sql_customer_totals.collect() == pyspark_customer_totals.collect()
show_df(sql_customer_totals)

## Step 7 - Alert Output and Supporting Transactions

An AML/TM alert should be explainable: deterministic key, rule metadata, observed value, threshold, and linked supporting transactions.

In [ ]:
alerts = (
    pyspark_customer_totals
    .withColumn("rule_id", F.lit("TM_HIGH_RISK_WIRE_001"))
    .withColumn("rule_version", F.lit("1.0.0"))
    .withColumn("processing_month", F.lit(processing_month))
    .withColumn("threshold_cad", F.lit(rule_threshold_cad).cast("decimal(18,2)"))
    .withColumn(
        "alert_key",
        F.sha2(F.concat_ws("|", "rule_id", "rule_version", "processing_month", "customer_id"), 256),
    )
)

supporting_transactions = (
    gold_rule_input.filter(F.col("risk_level") == "HIGH")
    .join(alerts.select("alert_key", "customer_id"), on="customer_id", how="inner")
    .select(
        "alert_key",
        "transaction_id",
        "customer_id",
        "account_id",
        "transaction_date",
        "amount_cad",
        "country_code",
        "risk_level",
    )
)

assert alerts.count() == 1
assert_set("alert customers", [r.customer_id for r in alerts.select("customer_id").collect()], ["c1"])
assert_set("supporting transaction ids", [r.transaction_id for r in supporting_transactions.select("transaction_id").collect()], ["t1", "t2"])
show_df(alerts.select("alert_key", "customer_id", "observed_amount_cad", "threshold_cad", "supporting_transaction_count"))
show_df(supporting_transactions.orderBy("transaction_id"))

## Step 8 - Reconciliation and Evidence Pack

Reconciliation is not a side report. It is evidence that rows moved as expected and that exceptions were visible.

In [ ]:
reconciliation = spark.createDataFrame([
    ("source_transactions", source_transactions.count()),
    ("bronze_transactions", bronze_transactions.count()),
    ("silver_transactions", silver_transactions.count()),
    ("june_posted_wires", june_posted_wires.count()),
    ("valid_customer_tx", valid_customer_tx.count()),
    ("orphan_accounts", orphan_accounts.count()),
    ("gold_rule_input", gold_rule_input.count()),
    ("high_risk_supporting_tx", supporting_transactions.count()),
    ("alerts", alerts.count()),
], ["step_name", "row_count"])

expected_counts = {
    "source_transactions": 8,
    "bronze_transactions": 8,
    "silver_transactions": 8,
    "june_posted_wires": 4,
    "valid_customer_tx": 3,
    "orphan_accounts": 1,
    "gold_rule_input": 3,
    "high_risk_supporting_tx": 2,
    "alerts": 1,
}
actual_counts = {r.step_name: r.row_count for r in reconciliation.collect()}
assert actual_counts == expected_counts, f"Expected {expected_counts}, got {actual_counts}"

evidence_manifest = spark.createDataFrame([
    (batch_id, processing_month, "rule_id", "TM_HIGH_RISK_WIRE_001"),
    (batch_id, processing_month, "rule_version", "1.0.0"),
    (batch_id, processing_month, "threshold_cad", str(rule_threshold_cad)),
    (batch_id, processing_month, "alert_count", str(alerts.count())),
    (batch_id, processing_month, "dq_orphan_count", str(orphan_accounts.count())),
], ["batch_id", "processing_month", "evidence_name", "evidence_value"])

show_df(reconciliation)
show_df(evidence_manifest.orderBy("evidence_name"))

## Step 9 - Delta-Style Persistence Demo

Databricks production pipelines usually write governed Delta tables. This cell tries Delta first and falls back to Parquet if the current environment does not have Delta configured. The learning point is the same: persist curated outputs and reload them for validation.

In [ ]:
if dbutils_ref is not None:
    demo_path = "dbfs:/tmp/aml_learning_for_fintech/databricks_one_stop_gold_rule_input"
    dbutils_ref.fs.rm(demo_path, True)
else:
    demo_path = "/tmp/aml_learning_for_fintech_databricks_one_stop_gold_rule_input"
    shutil.rmtree(demo_path, ignore_errors=True)

try:
    gold_rule_input.write.format("delta").mode("overwrite").save(demo_path)
    reloaded_gold_rule_input = spark.read.format("delta").load(demo_path)
    storage_format = "delta"
except Exception as delta_error:
    if dbutils_ref is not None:
        dbutils_ref.fs.rm(demo_path, True)
    else:
        shutil.rmtree(demo_path, ignore_errors=True)
    gold_rule_input.write.mode("overwrite").parquet(demo_path)
    reloaded_gold_rule_input = spark.read.parquet(demo_path)
    storage_format = f"parquet fallback after {delta_error.__class__.__name__}"

assert reloaded_gold_rule_input.count() == gold_rule_input.count()
print(f"Persisted and reloaded gold_rule_input using {storage_format} at {demo_path}")

## Step 10 - Databricks SQL / BI-Ready Views

Databricks SQL and BI tools should consume governed outputs, not raw ad hoc temp data. Here we create views that mirror alert, support, DQ, and reconciliation reporting surfaces.

In [ ]:
alerts.createOrReplaceTempView("gold_alerts")
supporting_transactions.createOrReplaceTempView("gold_alert_supporting_transactions")
dq_summary.createOrReplaceTempView("gold_dq_summary")
reconciliation.createOrReplaceTempView("gold_reconciliation")

dashboard_summary = spark.sql("""
SELECT 'alerts' AS metric_name, COUNT(*) AS metric_value FROM gold_alerts
UNION ALL
SELECT 'supporting_transactions' AS metric_name, COUNT(*) AS metric_value FROM gold_alert_supporting_transactions
UNION ALL
SELECT 'dq_checks_with_failures' AS metric_name, COUNT(*) AS metric_value FROM gold_dq_summary WHERE failed_row_count > 0
UNION ALL
SELECT 'reconciliation_steps' AS metric_name, COUNT(*) AS metric_value FROM gold_reconciliation
""")

assert {r.metric_name for r in dashboard_summary.collect()} == {
    "alerts",
    "supporting_transactions",
    "dq_checks_with_failures",
    "reconciliation_steps",
}
show_df(dashboard_summary.orderBy("metric_name"))

## Step 11 - Lakeflow / Jobs Thinking as a Runnable Plan

This notebook does not create Databricks Jobs or Lakeflow pipelines. Instead, it creates the task plan you should be able to explain before productionizing the notebook.

In [ ]:
job_task_plan = spark.createDataFrame([
    (1, "ingest_bronze", "Lakeflow Connect or ADF/Fabric landing task", "source checks and manifest"),
    (2, "standardize_silver", "Spark SQL or PySpark transformation", "schema, type, and key normalization checks"),
    (3, "run_dq", "Lakeflow expectations or explicit DQ queries", "exception tables and DQ summary"),
    (4, "build_gold_rule_input", "PySpark or Spark SQL rule-ready table", "row-count reconciliation"),
    (5, "execute_rule", "Databricks job task", "alerts and supporting transactions"),
    (6, "publish_evidence", "Databricks SQL / BI / audit pack task", "manifest, reconciliation, and sign-off artifacts"),
], ["task_order", "task_name", "databricks_pattern", "evidence_output"])

assert job_task_plan.count() == 6
show_df(job_task_plan.orderBy("task_order"))

## Step 12 - ML / Analytics Feature Readiness

ML in AML/TM should usually start as decision support. This feature table is not a model; it is a governed, explainable input that could support prioritization or false-positive analysis.

In [ ]:
customer_features = (
    gold_rule_input.groupBy("customer_id")
    .agg(
        F.count("*").alias("posted_wire_count"),
        F.sum("amount_cad").alias("posted_wire_amount_cad"),
        F.sum(F.when(F.col("risk_level") == "HIGH", 1).otherwise(0)).alias("high_risk_wire_count"),
        F.sum(F.when(F.col("risk_level") == "HIGH", F.col("amount_cad")).otherwise(F.lit(0))).alias("high_risk_wire_amount_cad"),
    )
)

alert_labels = alerts.select("customer_id").withColumn("alert_generated", F.lit(1))
feature_ready = customer_features.join(alert_labels, on="customer_id", how="left").fillna({"alert_generated": 0})

assert feature_ready.count() == 2
assert_set("feature customers", [r.customer_id for r in feature_ready.select("customer_id").collect()], ["c1", "c2"])
show_df(feature_ready.orderBy("customer_id"))

## Step 13 - Performance and Debugging Hooks

On Databricks, pair `explain` output with the Spark UI. Look for join strategy, shuffles, filters pushed before joins, skewed keys, and unnecessary wide transformations.

In [ ]:
print("Logical and physical plan for gold_rule_input:")
gold_rule_input.explain(True)

debug_counts = spark.createDataFrame([
    ("partitions_gold_rule_input", gold_rule_input.rdd.getNumPartitions()),
    ("partitions_alerts", alerts.rdd.getNumPartitions()),
], ["debug_metric", "debug_value"])
show_df(debug_counts)

## Step 14 - Tech Stack Micro-Lab: Same Rule in Spark SQL and PySpark

This cell group belongs here instead of Markdown because PySpark, Python, and Spark SQL learning should be runnable from a notebook. The goal is to prove that Spark SQL and PySpark can express the same AML/TM rule and produce the same expected rows.

Scenario: identify customers whose June 2022 posted transaction total is at least 10,000 CAD.

In [ ]:
micro_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("transaction_date", T.StringType(), False),
    T.StructField("status", T.StringType(), False),
    T.StructField("amount_cad", T.DoubleType(), False),
])

micro_rows = [
    ("T001", "C001", "2022-06-01", "POSTED", 6000.0),
    ("T002", "C001", "2022-06-10", "POSTED", 4500.0),
    ("T003", "C002", "2022-06-12", "POSTED", 3000.0),
    ("T004", "C002", "2022-06-20", "REVERSED", 9000.0),
    ("T005", "C003", "2022-07-01", "POSTED", 20000.0),
]

tech_stack_micro_transactions = spark.createDataFrame(micro_rows, micro_schema)
tech_stack_micro_transactions.createOrReplaceTempView("tech_stack_micro_transactions")

show_df(tech_stack_micro_transactions.orderBy("transaction_id"))
print("Expected input count: 5")
assert tech_stack_micro_transactions.count() == 5

### Step 14A - Run the Spark SQL Version

Expected output: one customer, `C001`, with June posted amount `10500.0`.

In [ ]:
tech_stack_sql_result = spark.sql("""
SELECT
    customer_id,
    ROUND(SUM(amount_cad), 2) AS june_posted_amount_cad
FROM tech_stack_micro_transactions
WHERE status = 'POSTED'
  AND transaction_date >= '2022-06-01'
  AND transaction_date < '2022-07-01'
GROUP BY customer_id
HAVING SUM(amount_cad) >= 10000
ORDER BY customer_id
""")

show_df(tech_stack_sql_result)

### Step 14B - Run the PySpark DataFrame Version

This should match the Spark SQL result exactly.

In [ ]:
tech_stack_pyspark_result = (
    tech_stack_micro_transactions
    .filter(
        (F.col("status") == "POSTED")
        & (F.col("transaction_date") >= "2022-06-01")
        & (F.col("transaction_date") < "2022-07-01")
    )
    .groupBy("customer_id")
    .agg(F.round(F.sum("amount_cad"), 2).alias("june_posted_amount_cad"))
    .filter(F.col("june_posted_amount_cad") >= 10000)
    .orderBy("customer_id")
)

show_df(tech_stack_pyspark_result)

### Step 14C - Validate SQL and PySpark Match

The assertions are the real learning contract: both implementations must produce the same expected business result.

In [ ]:
tech_stack_sql_rows = [row.asDict() for row in tech_stack_sql_result.collect()]
tech_stack_pyspark_rows = [row.asDict() for row in tech_stack_pyspark_result.collect()]
tech_stack_expected_rows = [{"customer_id": "C001", "june_posted_amount_cad": 10500.0}]

assert tech_stack_sql_rows == tech_stack_expected_rows, tech_stack_sql_rows
assert tech_stack_pyspark_rows == tech_stack_expected_rows, tech_stack_pyspark_rows
assert tech_stack_sql_rows == tech_stack_pyspark_rows

tech_stack_micro_lab_status = "PASS"
print("SQL and PySpark rule outputs match expected result.")

## Step 15 - Final Validation Scorecard

If this cell passes, the notebook ran end to end and produced the expected learning artifacts.

In [ ]:
scorecard = spark.createDataFrame([
    ("bronze_count", "PASS" if bronze_transactions.count() == 8 else "FAIL"),
    ("silver_count", "PASS" if silver_transactions.count() == 8 else "FAIL"),
    ("orphan_visible", "PASS" if orphan_accounts.count() == 1 else "FAIL"),
    ("gold_rule_input_count", "PASS" if gold_rule_input.count() == 3 else "FAIL"),
    ("alert_count", "PASS" if alerts.count() == 1 else "FAIL"),
    ("supporting_transaction_count", "PASS" if supporting_transactions.count() == 2 else "FAIL"),
    ("feature_ready_count", "PASS" if feature_ready.count() == 2 else "FAIL"),
    ("tech_stack_micro_lab", "PASS" if tech_stack_micro_lab_status == "PASS" else "FAIL"),
], ["validation_name", "test_status"])

assert scorecard.filter(F.col("test_status") != "PASS").count() == 0
show_df(scorecard.orderBy("validation_name"))
print("Databricks one-stop notebook validation passed.")

## Closed-Book Drills

1. Change `rule_threshold_cad` to `120`. Predict the alert count before rerunning.
2. Change `t6` account from `a9` to `a2`. Predict the orphan count, gold input count, and alert count.
3. Change `t2` country from `IR` to `CA`. Predict `high_risk_supporting_tx` and `alerts`.
4. Explain which notebook cells would become Databricks Jobs tasks.
5. Explain which outputs should become governed Delta tables in Unity Catalog.
6. Explain which validation checks belong in Lakeflow expectations, explicit DQ tables, and CI.
7. In Step 14, change the micro-lab threshold from `10000` to `11000`. Predict whether `C001` still appears before rerunning.